In [1]:
from google.colab import auth
auth.authenticate_user()

In [23]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import re
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [24]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/REPORTES DE ERRORES/CONTICASA/ERRORES-RESPUESTA-TICKET/"

DATASET_ID = "produccion"
TABLE_ID= "Errores_CONTICASA"

#FECHA_PERIODO= "2025-09-05"
#PROJECT_ID = "test-proyect-468615"
#BUCKET_NAME = "data_bucket_proy"
#FOLDER_PATH= "desgravamen_prestamos/"
#DATASET_ID = "db_test"
#TABLE_ID= "desgravamen_prestamos"


# SCRIPT COMPLETO

--------------

In [25]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [26]:
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

In [27]:
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    print(blob.name)
    #print(excel_file.sheet_names)

data_entries/REPORTES DE ERRORES/CONTICASA/ERRORES-RESPUESTA-TICKET/Consolidado_errores_conticasa_enero25-julio25.xlsx


In [28]:
print(blobs_excels[0].name)
file = blobs_excels[0].download_as_string()
df_conticasa= pd.read_excel(BytesIO(file), sheet_name='Hoja1', dtype={'CODPROD':str, 'NUMPOL':str,'NUMCERT':str,'NRO_CERT_BANCO': str, 'CANTPROR': str} )

data_entries/REPORTES DE ERRORES/CONTICASA/ERRORES-RESPUESTA-TICKET/Consolidado_errores_conticasa_enero25-julio25.xlsx


In [29]:
df_conticasa.head(3)

,MES,CODPROD,NUMPOL,NUMCERT,NRO_CERT_BANCO,CANTPROR,INDICADOR,OBSERVACION
0,2025-01-01,1304,501486,35,00110196354000274731,2,-1,ORA-20100: PR_POLIZA.ACTIVAR: ORA-20100: PR_PO...
1,2025-01-01,1304,501490,161,00110482794000320444,2,-1,ORA-20100: PR_POLIZA.ACTIVAR: ORA-20100: PR_PO...
2,2025-01-01,1304,501490,162,00110320964000396317,2,-1,ORA-20100: PR_POLIZA.ACTIVAR: ORA-20100: PR_PO...


In [30]:
df_conticasa= df_conticasa.fillna({'CODPROD':'', 'NUMPOL': '', 'NUMCERT': '', 'NRO_CERT_BANCO': '', 'CANTPROR': '0'})

In [31]:
df_conticasa['POLIZA_Y_CERTIFICADO']= df_conticasa['CODPROD'] + '-'+ df_conticasa['NUMPOL'] + '-' + df_conticasa['NUMCERT']

In [32]:
def separar_Observacion(observacion, indicador):
    resultado = []
    if indicador == -1:
        partes = observacion.split(':')
        nro_partes= len(partes)
        if nro_partes == 1:
            resultado= [partes[0],"",""]
        elif nro_partes == 2:
            resultado= [partes[1],"",""]
        elif nro_partes == 4:
            resultado = [partes[1], "", partes[3]]
        else:
            resultado = [partes[1], partes[3], ':'.join(partes[4:])]
    else:
        partes = observacion.split('|')
        if len(partes) == 1:
            resultado = [partes[0], "", ""]
        else:
            resultado = [partes[0], partes[1], ""]
    return resultado

In [33]:
# Aplicar la función y expandir en columnas
df_conticasa[['DESCRIPCION1', 'DESCRIPCION2', 'DATALLE']] = df_conticasa.apply(lambda row: pd.Series(separar_Observacion(row['OBSERVACION'], row['INDICADOR'])), axis=1)

In [34]:
df_conticasa["POLIZA_Y_CERTIFICADO"] = df_conticasa["POLIZA_Y_CERTIFICADO"].astype(str)

In [35]:
df_conticasa.head()

,MES,CODPROD,NUMPOL,NUMCERT,NRO_CERT_BANCO,CANTPROR,INDICADOR,OBSERVACION,POLIZA_Y_CERTIFICADO,DESCRIPCION1,DESCRIPCION2,DATALLE
0,2025-01-01,1304,501486,35,00110196354000274731,2,-1,ORA-20100: PR_POLIZA.ACTIVAR: ORA-20100: PR_PO...,1304-501486-35,PR_POLIZA.ACTIVAR,PR_POLIZA.ACTIVAR,Existen certificados que no tienen configurad...
1,2025-01-01,1304,501490,161,00110482794000320444,2,-1,ORA-20100: PR_POLIZA.ACTIVAR: ORA-20100: PR_PO...,1304-501490-161,PR_POLIZA.ACTIVAR,PR_POLIZA.ACTIVAR,Existen certificados que no tienen configurad...
2,2025-01-01,1304,501490,162,00110320964000396317,2,-1,ORA-20100: PR_POLIZA.ACTIVAR: ORA-20100: PR_PO...,1304-501490-162,PR_POLIZA.ACTIVAR,PR_POLIZA.ACTIVAR,Existen certificados que no tienen configurad...
3,2025-01-01,1304,501490,164,00110566764000303139,2,-1,ORA-20100: PR_POLIZA.ACTIVAR: ORA-20100: PR_PO...,1304-501490-164,PR_POLIZA.ACTIVAR,PR_POLIZA.ACTIVAR,Existen certificados que no tienen configurad...
4,2025-01-01,1304,501490,179,00110615234000170150,1,-1,ORA-20100: PR_POLIZA.ACTIVAR: ORA-20100: PR_PO...,1304-501490-179,PR_POLIZA.ACTIVAR,PR_POLIZA.ACTIVAR,Existen certificados que no tienen configurad...


In [36]:
df_conticasa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4712 entries, 0 to 4711
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   MES                   4712 non-null   datetime64[ns]
 1   CODPROD               4712 non-null   object        
 2   NUMPOL                4712 non-null   object        
 3   NUMCERT               4712 non-null   object        
 4   NRO_CERT_BANCO        4712 non-null   object        
 5   CANTPROR              4712 non-null   object        
 6   INDICADOR             4712 non-null   int64         
 7   OBSERVACION           4712 non-null   object        
 8   POLIZA_Y_CERTIFICADO  4712 non-null   object        
 9   DESCRIPCION1          4712 non-null   object        
 10  DESCRIPCION2          4712 non-null   object        
 11  DATALLE               4712 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 441.9+ KB


In [37]:
#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

In [40]:
schema_conticasa = [
        bigquery.SchemaField("MES", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODPROD", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMPOL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMCERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_CERT_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CANTPROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("INDICADOR", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("OBSERVACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("POLIZA_Y_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION1", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION2", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DATALLE", bigquery.enums.SqlTypeNames.STRING)
    ]
Guardar_en_BigQuery(df_conticasa, DATASET_ID, TABLE_ID, schema_conticasa)

----- Se ha creado la tabla Errores_CONTICASA en el dataset produccion -----
